# FlowerClf — emit submission from precomputed probs

Inference already ran on the local RTX 3090 Ti (`src/train.py`). This notebook just loads
the saved test softmax probs (attached dataset), argmax → label, and writes `submission.csv`
ordered to match `sample_submission.csv`. No model / GPU needed — seconds to run.

In [ ]:
import os, glob
import numpy as np, pandas as pd
print('inputs:', os.listdir('/kaggle/input'))

probs_path = glob.glob('/kaggle/input/**/*_test_probs.npy', recursive=True)[0]
ids_path = glob.glob('/kaggle/input/**/*_test_ids.npy', recursive=True)[0]
sample_path = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)[0]
print('probs:', probs_path); print('ids:', ids_path)

probs = np.load(probs_path)
ids = np.load(ids_path, allow_pickle=True).astype(str)
print('probs', probs.shape, 'ids', ids.shape)

In [ ]:
sample = pd.read_csv(sample_path)
sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)})
assert set(sub['id']) == set(sample['id'].astype(str)), 'id mismatch vs sample_submission'
sub = sub.set_index('id').loc[sample['id'].astype(str)].reset_index()
assert sub['label'].between(0, 103).all()
sub.to_csv('submission.csv', index=False)
print('wrote submission.csv', sub.shape, 'classes used:', sub['label'].nunique())
sub.head()